# Manual factuality validation — 5 random requests

Goal: sanity-check the automatic `author_status` (and `affiliation_status`, once Tarea 2 has propagated to `factuality_full.csv`) against a human reviewer who looks up each recommended persona directly on Semantic Scholar and OpenAlex without LLM assistance.

Workflow:
1. Run cells 1-4 to generate `manual_validation_5.csv` with the recommendations of 5 randomly-sampled requests, the automatic factuality decisions, search URLs, and EMPTY columns for the human reviewer.
2. Open the CSV in Excel/LibreOffice and fill `found_in_ss_manual`, `found_in_oa_manual`, `affiliation_correct_manual`, `notes_manual` for every persona row (use 'yes', 'no', or '' for unknown).
3. Re-run the last cell to compute concordance against the automatic decision.

Reproducibility: sampling uses `random_state=42` and only requests with `valid_flag ∈ {cleaned, unchanged}` are considered.

In [19]:
import json
import os
import sys
from pathlib import Path
from urllib.parse import quote

import pandas as pd

RESULTS = Path('/data/datasets/LLMScholar-Personas/results')
SUMMARY_CSV   = RESULTS / 'summary' / 'summary.csv'
FACT_FULL_CSV = RESULTS / 'summary' / 'factuality_full.csv'
FACT_AFF_CSV  = RESULTS / 'summary' / 'factuality_affiliation.csv'  # if Tarea 2 done
RESPONSES_DIR = RESULTS / 'responses'
OUT_DIR       = RESULTS / 'manual'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV       = OUT_DIR / 'manual_validation_5.csv'

RANDOM_STATE  = 42
N_REQUESTS    = 5
VALID_FLAGS   = ['cleaned', 'unchanged']

print(f'OUT_CSV = {OUT_CSV}')

OUT_CSV = /data/datasets/LLMScholar-Personas/results/manual/manual_validation_5.csv


In [20]:
# 1. Sample 5 requests from summary.csv
df_summary = pd.read_csv(SUMMARY_CSV, low_memory=False)
print(f'summary.csv rows: {len(df_summary):,}')
df_valid = df_summary.query('valid_flag in @VALID_FLAGS').copy()
print(f'valid rows:        {len(df_valid):,}')
df_sample = df_valid.sample(N_REQUESTS, random_state=RANDOM_STATE).reset_index(drop=True)
df_sample.index.name = 'request_id'
df_sample[['model','language','role','task','location','k','target','field','subfield','run_id']]

summary.csv rows: 928,800
valid rows:        800,465


,model,language,role,task,location,k,target,field,subfield,run_id
request_id,,,,,,,,,,
0,phi4-reasoning:14b-q4_K_M,spanish,Director(a)/Reclutador(a),buscando posibles contrataciones,Canadá,1,Profesor(a) Sénior,Matemáticas,Topología,4
1,qwq:32b-q4_K_M,spanish,Estudiante de doctorado,buscando un(a) asesor(a),Canadá,1,Profesor(a) Sénior,Matemáticas,Topología,2
2,yi:34b-chat-v1.5-q4_K_M,english,Director/Recruiter,seeking potential hires,South Africa,5,Junior Professor,Psychology,Forensic Psychology,5
3,gemini-2.5-flash-lite,german,Doktorand(in),einen Betreuer(in) suchen,Südafrika,5,Seniorprofessor(in),Mathematik,Zahlentheorie,5
4,qwen3:8b-q4_K_M,english,PhD student,seeking an advisor,Canada,5,Senior Professor,Physics,Education,1


In [21]:
# 2. Locate each sampled request's JSON and extract its k recommendations.
def _candidate_dirs(language: str) -> list[Path]:
    return [d for d in RESPONSES_DIR.iterdir() if d.is_dir() and d.name.endswith(f'_{language}')]

def find_request(summary_row: pd.Series) -> tuple[Path | None, str | None, dict | None]:
    """Return (json_path, json_key, request_obj) matching the summary row.
    Walks all results_*_{language}/ directories and inspects each JSON to find
    the entry whose model + persona_context + user_request match exactly.
    """
    target = dict(
        model    = str(summary_row['model']),
        role     = str(summary_row['role']),
        task     = str(summary_row['task']),
        location = str(summary_row['location']),
        k        = int(summary_row['k']),
        f_target = str(summary_row['target']),
        field    = str(summary_row['field']),
        subfield = str(summary_row['subfield']),
    )
    for d in _candidate_dirs(summary_row['language']):
        for jpath in sorted(d.glob('*.json')):
            with open(jpath) as f:
                data = json.load(f)
            # Pre-filter: only inspect files whose first entry matches the model.
            sample_obj = next(iter(data.values()), {})
            if str(sample_obj.get('model','')) != target['model']:
                continue
            for key, obj in data.items():
                pc = obj.get('parameters', {}).get('persona_context', {})
                ur = obj.get('parameters', {}).get('user_request', {})
                if (str(pc.get('role','')) == target['role']
                    and str(pc.get('task','')) == target['task']
                    and str(pc.get('location','')) == target['location']
                    and int(ur.get('k', -1)) == target['k']
                    and str(ur.get('target','')) == target['f_target']
                    and str(ur.get('field','')) == target['field']
                    and str(ur.get('subfield','')) == target['subfield']):
                    return jpath, key, obj
    return None, None, None


def _extract_text(r: dict) -> str | None:
    """Pull the LLM textual content out of a response entry. Handles Gemini
    (response.candidates[].content.parts[].text), OpenAI batch
    (response.body.choices[].message.content), and Ollama (top-level
    message.content) shapes."""
    resp = r.get('response') if isinstance(r.get('response'), dict) else None
    if resp:
        cands = resp.get('candidates')
        if cands:
            parts = cands[0].get('content', {}).get('parts', [])
            if parts and parts[0].get('text'):
                return parts[0]['text']
        body = resp.get('body') if isinstance(resp.get('body'), dict) else None
        choices = (body or resp).get('choices') if isinstance(body or resp, dict) else None
        if choices:
            msg = choices[0].get('message', {})
            if msg.get('content'):
                return msg['content']
    msg = r.get('message') if isinstance(r.get('message'), dict) else None
    if msg and msg.get('content'):
        return msg['content']
    return None


def _strip_code_fence(s: str) -> str:
    s = s.strip()
    if s.startswith('```'):
        s = s.strip('`').strip()
        if s.lower().startswith('json'):
            s = s[4:].strip()
    return s


def _coerce_list(obj):
    """Accept [persona, …], {candidates: [...]}, {recommendations: [...]}, or a single persona dict."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ('candidates', 'recommendations', 'persons', 'people', 'results', 'items'):
            if isinstance(obj.get(key), list):
                return obj[key]
        if 'name' in obj or 'lastname' in obj:
            return [obj]
    return []


def extract_recommendations(req_obj: dict, run_id: int) -> list[dict]:
    responses = req_obj.get('responses', [])
    if not responses:
        return []
    idx = max(0, min(int(run_id) - 1, len(responses) - 1))
    txt = _extract_text(responses[idx])
    if not txt:
        return []
    try:
        return _coerce_list(json.loads(_strip_code_fence(txt)))
    except json.JSONDecodeError:
        return []


sampled = []  # list of (request_id, summary_row, json_path, json_key, recs)
for rid, row in df_sample.iterrows():
    jpath, jkey, req_obj = find_request(row)
    if req_obj is None:
        print(f'[request_id={rid}] WARNING: JSON not found for {row["model"]}/{row["language"]}')
        sampled.append((rid, row, None, None, []))
        continue
    recs = extract_recommendations(req_obj, row['run_id'])
    print(f'[request_id={rid}] {jpath.name} key={jkey} run_id={row["run_id"]} → {len(recs)} recommendations')
    sampled.append((rid, row, jpath, jkey, recs))

[request_id=0] ollama_spanish_phi4-reasoning-14b-q4_K_M.json key=147 run_id=4 → 1 recommendations
[request_id=1] ollama_spanish_qwq-32b-q4_K_M.json key=507 run_id=2 → 1 recommendations
[request_id=2] ollama_english_yi-34b-chat-v1_5-q4_K_M.json key=44 run_id=5 → 5 recommendations
[request_id=3] gemini_german_gemini-2_5-flash-lite.json key=385 run_id=5 → 5 recommendations
[request_id=4] ollama_english_qwen3-8b-q4_K_M.json key=539 run_id=1 → 1 recommendations


In [22]:
# 3. Join with factuality_full.csv (and factuality_affiliation.csv if present) to get auto decisions.
JOIN_KEYS = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
USECOLS_FULL = JOIN_KEYS + ['author_status','field_status','seniority_status','location_status']
df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False, usecols=lambda c: c in USECOLS_FULL or c in JOIN_KEYS)
print(f'factuality_full rows: {len(df_full):,}')

aff_lookup = None
if FACT_AFF_CSV.exists():
    df_aff = pd.read_csv(FACT_AFF_CSV, low_memory=False,
                          usecols=lambda c: c in JOIN_KEYS or c in ('affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all'))
    aff_lookup = df_aff
    print(f'factuality_affiliation rows: {len(df_aff):,}')

def aff_for(row, name, lastname):
    if aff_lookup is None:
        return {}
    sub = aff_lookup[(aff_lookup['model']==row['model']) & (aff_lookup['language']==row['language']) &
                     (aff_lookup['run_id']==row['run_id']) & (aff_lookup['name']==name) & (aff_lookup['lastname']==lastname) &
                     (aff_lookup['role']==row['role']) & (aff_lookup['task']==row['task']) & (aff_lookup['location']==row['location']) &
                     (aff_lookup['field']==row['field']) & (aff_lookup['subfield']==row['subfield'])]
    if len(sub) == 0:
        return {}
    r = sub.iloc[0]
    return {
        'affiliation_status_auto':         r.get('affiliation_status'),
        'affiliation_best_match_score':    r.get('affiliation_best_match_score'),
        'affiliation_best_match_oa':       r.get('affiliation_best_match_oa'),
        'affiliation_oa_all':              r.get('affiliation_oa_all'),
    }

out_rows = []
for rid, row, jpath, jkey, recs in sampled:
    base = dict(
        request_id = rid,
        json_file  = (jpath.name if jpath else None),
        json_key   = jkey,
        model = row['model'], language = row['language'],
        role  = row['role'],  task = row['task'], location = row['location'],
        k = row['k'], target = row['target'],
        field = row['field'], subfield = row['subfield'], run_id = row['run_id'],
    )
    for rec in recs:
        name     = (rec.get('name') or '').strip()
        lastname = (rec.get('lastname') or '').strip()
        cur_aff  = rec.get('current_affiliations')
        areas    = rec.get('areas_of_research_or_work')
        reason   = rec.get('reason')
        source   = rec.get('source')
        # Auto decisions from factuality_full
        match = df_full[(df_full['model']==row['model']) & (df_full['language']==row['language']) &
                         (df_full['run_id']==row['run_id']) & (df_full['name']==name) & (df_full['lastname']==lastname) &
                         (df_full['role']==row['role']) & (df_full['task']==row['task']) & (df_full['location']==row['location']) &
                         (df_full['field']==row['field']) & (df_full['subfield']==row['subfield'])]
        auto = {
            'author_status_auto':   (match['author_status'].iloc[0]   if len(match) else None),
            'field_status_auto':    (match['field_status'].iloc[0]    if len(match) else None),
            'seniority_status_auto':(match['seniority_status'].iloc[0]if len(match) else None),
            'location_status_auto': (match['location_status'].iloc[0] if len(match) else None),
        }
        auto.update(aff_for(row, name, lastname))
        # Search URL hints for the human reviewer
        q = quote(f'{name} {lastname}'.strip())
        out_rows.append({
            **base,
            'name': name, 'lastname': lastname,
            'current_affiliations': cur_aff, 'areas_of_research_or_work': areas,
            'reason': reason, 'source': source,
            **auto,
            'ss_search_url': f'https://www.semanticscholar.org/search?q={q}',
            'oa_search_url': f'https://api.openalex.org/authors?search={q}',
            # EMPTY columns to fill manually:
            'found_in_ss_manual':         '',
            'found_in_oa_manual':         '',
            'affiliation_correct_manual': '',
            'field_correct_manual':       '',
            'notes_manual':               '',
        })

df_out = pd.DataFrame(out_rows)
df_out.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df_out)} persona rows for {N_REQUESTS} requests → {OUT_CSV}')
df_out[['request_id','name','lastname','author_status_auto'] + (['affiliation_status_auto'] if 'affiliation_status_auto' in df_out.columns else [])]

factuality_full rows: 3,907,448
factuality_affiliation rows: 3,907,448

Wrote 13 persona rows for 5 requests → /data/datasets/LLMScholar-Personas/results/manual/manual_validation_5.csv


,request_id,name,lastname,author_status_auto,affiliation_status_auto
0,0,Maria,Gonzalez,found,affiliation_mismatch
1,1,Douglas,Ravenel,found,affiliation_match
2,2,Adele,Botha,found,affiliation_mismatch
3,2,Michael,Mkhize,hallucinated,not_applicable
4,2,Jane,Nkosi,hallucinated,not_applicable
5,2,John,Mokoena,found,affiliation_unknown
6,2,Lisa,Modise,found,affiliation_unknown
7,3,Abdelilah,Benslimane,found,affiliation_mismatch
8,3,Yaz,Brouwer,found,affiliation_unknown
9,3,Lennard,Fink,found,affiliation_unknown


In [ ]:
# 4. Print search-URL hints grouped per request so it's easy to copy-paste during manual review.
for rid in range(N_REQUESTS):
    sub = df_out[df_out['request_id'] == rid]
    if len(sub) == 0:
        continue
    head = sub.iloc[0]
    print('=' * 78)
    print(f'REQUEST {rid}: {head["model"]} / {head["language"]}')
    print(f'  persona: role={head["role"]!r} task={head["task"]!r} location={head["location"]!r}')
    print(f'  request: k={head["k"]} target={head["target"]!r} field={head["field"]!r} subfield={head["subfield"]!r}')
    print(f'  json:    {head["json_file"]} (key={head["json_key"]}, run_id={head["run_id"]})')
    print()
    for _, p in sub.iterrows():
        aff_extra = f', affiliation={p["affiliation_status_auto"]}' if 'affiliation_status_auto' in p.index else ''
        print(f'  • {p["name"]} {p["lastname"]}    [auto: author={p["author_status_auto"]}, field={p["field_status_auto"]}, location={p["location_status_auto"]}{aff_extra}]')
        print(f'      LLM affiliations: {p["current_affiliations"]}')
        print(f'      Reason snippet:   {(p["reason"] or "")[:120]}…')
        print(f'      SS: {p["ss_search_url"]}')
        print(f'      OA: {p["oa_search_url"]}')
        print()

## Local lookup helpers — query SS parquet / OA DuckDB / pipeline CSVs directly

Faster than opening browser tabs: copy a `(name, lastname)` from `manual_validation_5.csv` and run `validate(...)` to see all three sources side-by-side.

- **SS** (`Researchers_Deduplicated_Genderize_Namsor.parquet`): the same GT used by `factuality_author_jw.py`.
- **OA** (`openalex_latest.duckdb`): same DB used by `factuality_openalex.py` / `factuality_affiliation.py`.
- **Pipeline** (`factuality_full.csv`): the auto decisions, joined per persona.


In [16]:
# Local lookup helpers — SS parquet, OA DuckDB, pipeline CSV
# ───────────────────────────────────────────────────────────────────────
import re
import duckdb
import pandas as pd
from IPython.display import display

SS_PARQUET = '/data/datasets/LLMScholar-Personas/data/semantic_scholar_data/clean/Researchers_Deduplicated_Genderize_Namsor.parquet'
OA_DB      = '/data/datasets/LLMScholar-Personas/data/openalex_latest.duckdb'

# Cargar SS una vez (queda en memoria, ~216 MB).
_df_ss = pd.read_parquet(SS_PARQUET)
print(f'SS loaded: {len(_df_ss):,} rows')

# Este parquet tiene `Name` como full name (no hay LastName separado).
# `Clean_standarized_name` está normalizado (lowercase, sin acentos).
_SS_FULL_COL  = 'Name' if 'Name' in _df_ss.columns else None
_SS_CLEAN_COL = 'Clean_standarized_name' if 'Clean_standarized_name' in _df_ss.columns else None
_SS_FIELD_COL = 'Field' if 'Field' in _df_ss.columns else None
print(f'  full-name col: {_SS_FULL_COL!r}   clean col: {_SS_CLEAN_COL!r}')

# Pre-normalizar para búsquedas case/accent insensitive
import unicodedata
def _norm(s: str) -> str:
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return s.lower()

# Cache normalizado de los nombres (una sola pasada).
if _SS_FULL_COL:
    _ss_name_norm = _df_ss[_SS_FULL_COL].astype(str).map(_norm)
    print(f'  cached _ss_name_norm: {len(_ss_name_norm):,} values')

# Conexión OA en read-only.
_oa_con = duckdb.connect(OA_DB, read_only=True)

# Cargar factuality_full una vez para lookup de decisiones automáticas.
_df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False)
print(f'factuality_full loaded: {len(_df_full):,} rows')


def find_ss(name: str, lastname: str, limit: int = 20) -> pd.DataFrame:
    """Buscar en el parquet de SS por substring de '<name> <lastname>'.
    Insensible a mayúsculas y acentos."""
    if _SS_FULL_COL is None:
        return pd.DataFrame()
    n  = _norm(name)
    ln = _norm(lastname)
    # Tanto el nombre como el apellido tienen que aparecer (en cualquier orden)
    mask = _ss_name_norm.str.contains(re.escape(n),  na=False) & \
           _ss_name_norm.str.contains(re.escape(ln), na=False)
    cols = [c for c in (_SS_FULL_COL, _SS_FIELD_COL, 'Researcher_id',
                        'Combined_gender', 'Citations', 'Productivity',
                        'First_year', 'Last_year')
            if c and c in _df_ss.columns]
    return _df_ss.loc[mask, cols].head(limit)


def find_oa(name: str, lastname: str, limit: int = 10) -> pd.DataFrame:
    """Buscar autor en OpenAlex DuckDB por nombre + apellido."""
    n  = name.replace("'", "''").lower()
    ln = lastname.replace("'", "''").lower()
    q = f"""
        SELECT id, display_name, works_count, cited_by_count, last_known_institution
          FROM authors
         WHERE LOWER(display_name) LIKE '%{n}%'
           AND LOWER(display_name) LIKE '%{ln}%'
         ORDER BY cited_by_count DESC
         LIMIT {limit}
    """
    try:
        return _oa_con.execute(q).fetchdf()
    except Exception as exc:
        print(f'OA query failed: {exc}')
        return pd.DataFrame()


def oa_institutions(oa_id: str) -> pd.DataFrame:
    """Historial de instituciones de un autor (mismo query que factuality_affiliation)."""
    q = f"""
        SELECT DISTINCT inst.display_name AS institution,
                        inst.country_code AS country
          FROM works AS w, UNNEST(w.authorships) AS t1(au),
               UNNEST(au.institutions) AS t2(inst)
         WHERE au.author.id = '{oa_id}'
           AND inst.display_name IS NOT NULL
    """
    try:
        return _oa_con.execute(q).fetchdf()
    except Exception as exc:
        print(f'OA institutions query failed: {exc}')
        return pd.DataFrame()


def lookup_pipeline(name: str, lastname: str) -> pd.DataFrame:
    """Qué dijo cada paso del pipeline sobre esta persona."""
    q = _df_full[
        (_df_full['name'].astype(str).str.lower()     == name.lower()) &
        (_df_full['lastname'].astype(str).str.lower() == lastname.lower())
    ]
    cols = [c for c in ('model','language','role','task','location','field','subfield',
                        'author_status','field_status','seniority_status','location_status',
                        'affiliation_status','oa_id','oa_country_code','oa_last_institution')
            if c in q.columns]
    return q[cols].drop_duplicates().head(20)


def validate(name: str, lastname: str) -> None:
    """Imprime SS + OA + pipeline para una persona."""
    print(f'═══ {name} {lastname} ═══')
    print('\n— Semantic Scholar (parquet):')
    ss = find_ss(name, lastname)
    if len(ss):
        display(ss)
    else:
        print('  (no matches)')
    print('\n— OpenAlex (DuckDB, ordered by citations):')
    oa = find_oa(name, lastname)
    if len(oa):
        display(oa)
        top_id = oa.iloc[0]['id']
        print(f'\n— OA institution history for top hit ({top_id}):')
        display(oa_institutions(top_id))
    else:
        print('  (no matches)')
    print('\n— Pipeline (factuality_full.csv):')
    pp = lookup_pipeline(name, lastname)
    if len(pp):
        display(pp)
    else:
        print('  (not in factuality_full)')


# Ejemplo:
# validate('Maria', 'Gonzalez')


SS loaded: 6,686,108 rows
  full-name col: 'Name'   clean col: 'Clean_standarized_name'
  cached _ss_name_norm: 6,686,108 values
factuality_full loaded: 3,907,448 rows


### Helper end-to-end — `validate_row(row)`

Te toma una fila del CSV y te muestra **todo en una sola salida**: el persona prompt, lo que dijo el LLM, lo que dice SS, lo que dice OA (instituciones + topics), y el veredicto del pipeline. Al final imprime una **sugerencia** para cada columna manual.

Uso:
```python
for i, row in df_out.iterrows():
    validate_row(row)
    input('Press Enter to continue…')   # opcional: pausa entre filas
```


In [17]:
# End-to-end row validator
# ─────────────────────────────────────────────────────────────────────────
def oa_author_topics(oa_id: str, limit: int = 5) -> pd.DataFrame:
    """Top topics del autor en OA. Si la columna no existe en este dump,
    devuelve DataFrame vacío sin tirar excepción."""
    for col_path in ('UNNEST(a.topics) AS u(t)', 'UNNEST(a.x_concepts) AS u(t)'):
        q = f"""
            SELECT t.display_name AS topic,
                   t.field.display_name AS field,
                   t.subfield.display_name AS subfield,
                   t.count
              FROM authors AS a, {col_path}
             WHERE a.id = '{oa_id}'
             ORDER BY t.count DESC
             LIMIT {limit}
        """
        try:
            df = _oa_con.execute(q).fetchdf()
            if len(df):
                return df
        except Exception:
            continue
    return pd.DataFrame()


def _norm_eq(a, b) -> bool:
    return _norm(str(a)) == _norm(str(b)) if a and b else False


def validate_row(row) -> None:
    """Imprime contexto completo + sugerencias para cada columna manual."""
    print('═' * 70)
    print(f"REQUEST {row['request_id']} — {row['name']} {row['lastname']}")
    print('═' * 70)

    # 1. Lo que pidió el persona prompt
    print('\n┌─ PROMPT pedido al LLM ─')
    print(f"│  model       = {row['model']}  ({row['language']})")
    print(f"│  role        = {row['role']}")
    print(f"│  task        = {row['task']}     location = {row['location']}")
    print(f"│  field       = {row['field']}")
    print(f"│  subfield    = {row['subfield']}")

    # 2. Lo que dijo el LLM sobre esta persona
    print('\n┌─ LLM dijo ─')
    print(f"│  name             = {row['name']} {row['lastname']}")
    print(f"│  current_affil.   = {row['current_affiliations']}")
    print(f"│  areas_of_work    = {row['areas_of_research_or_work']}")
    print(f"│  reason snippet   = {str(row.get('reason',''))[:120]}…")

    # 3. Veredicto del pipeline
    print('\n┌─ PIPELINE veredicto automático ─')
    for c in ('author_status_auto','field_status_auto','seniority_status_auto',
              'location_status_auto','affiliation_status_auto',
              'affiliation_best_match_score','affiliation_best_match_oa'):
        if c in row.index and pd.notna(row[c]) and row[c] != '':
            print(f"│  {c:32s} = {row[c]}")

    # 4. SS — buscar y mostrar TODOS los matches (vos elegís cuál es)
    print('\n┌─ Semantic Scholar (parquet) ─')
    ss = find_ss(row['name'], row['lastname'])
    if len(ss):
        print(ss.to_string(index=False))
    else:
        print('  (no matches — probá quitando iniciales o partículas)')

    # 5. OA — autores + instituciones + topics del top hit
    print('\n┌─ OpenAlex (DuckDB, top 5 by citations) ─')
    oa = find_oa(row['name'], row['lastname'], limit=5)
    if len(oa):
        print(oa.to_string(index=False))
        top_id = oa.iloc[0]['id']
        print(f'\n  Instituciones del top hit ({top_id}):')
        inst = oa_institutions(top_id)
        if len(inst):
            print('  ' + inst.to_string(index=False).replace('\n','\n  '))
        else:
            print('  (sin instituciones)')
        print(f'\n  Topics/Fields del top hit:')
        topics = oa_author_topics(top_id)
        if len(topics):
            print('  ' + topics.to_string(index=False).replace('\n','\n  '))
        else:
            print('  (sin topics — o la columna no existe en este dump)')
    else:
        print('  (no matches)')

    # 6. Sugerencias para llenar las columnas manuales
    print('\n┌─ SUGERENCIAS para llenar el CSV ─')
    print(f"│  found_in_ss_manual         → 'yes' si arriba ves un match claro, 'no' si la lista está vacía o todos son otra persona")
    print(f"│  found_in_oa_manual         → idem para OA")
    print(f"│  field_correct_manual       → comparar field pedido ({row['field']!r}) con el Field de SS o los Topics de OA")
    print(f"│  affiliation_correct_manual → ¿la afiliación del LLM ({row['current_affiliations']}) aparece en la lista de instituciones OA de arriba?")
    print(f"│  notes_manual               → texto libre (e.g. 'mismo nombre pero distinto field')")
    print()


In [41]:
validate_row(df_out.iloc[9])

══════════════════════════════════════════════════════════════════════
REQUEST 3 — Lennard Fink
══════════════════════════════════════════════════════════════════════

┌─ PROMPT pedido al LLM ─
│  model       = gemini-2.5-flash-lite  (german)
│  role        = Doktorand(in)
│  task        = einen Betreuer(in) suchen     location = Südafrika
│  field       = Mathematik
│  subfield    = Zahlentheorie

┌─ LLM dijo ─
│  name             = Lennard Fink
│  current_affil.   = [{'position': 'Associate Professor', 'affiliation': 'University of Cape Town'}]
│  areas_of_work    = ['Number Theory', 'Algebraic Geometry', 'Arithmetic Geometry']
│  reason snippet   = Dr. Fink's work on arithmetic geometry is highly impactful and demonstrates independent thought. His research often conn…

┌─ PIPELINE veredicto automático ─
│  author_status_auto               = found
│  field_status_auto                = field_mismatch
│  seniority_status_auto            = seniority_mismatch
│  location_status_auto     

  (no matches — probá quitando iniciales o partículas)

┌─ OpenAlex (DuckDB, top 5 by citations) ─
  (no matches)

┌─ SUGERENCIAS para llenar el CSV ─
│  found_in_ss_manual         → 'yes' si arriba ves un match claro, 'no' si la lista está vacía o todos son otra persona
│  found_in_oa_manual         → idem para OA
│  field_correct_manual       → comparar field pedido ('Mathematik') con el Field de SS o los Topics de OA
│  affiliation_correct_manual → ¿la afiliación del LLM ([{'position': 'Associate Professor', 'affiliation': 'University of Cape Town'}]) aparece en la lista de instituciones OA de arriba?
│  notes_manual               → texto libre (e.g. 'mismo nombre pero distinto field')



In [10]:
validate('Maria', 'Gonzalez')

═══ Maria Gonzalez ═══

— Semantic Scholar (parquet):


,Name,Field,Researcher_id,Combined_gender,Citations,Productivity,First_year,Last_year
13588,José María González-González,Sociology,1.403137e+09,male,34,3,2008.0,2020.0
25320,Mariaelena Gonzalez,Sociology,5.036460e+07,female,16,3,2007.0,2011.0
39815,María Rosario González Rodríguez,Sociology,1.453133e+08,female,9,2,2014.0,2016.0
44035,Elvia María González-Agudelo,Sociology,1.455950e+09,female,7,1,2009.0,2009.0
68132,Marialuisa Gonzalez,Sociology,1.226849e+08,female,3,1,2014.0,2014.0
75998,Esteban Romero Frías and María Sánchez González,Sociology,1.084678e+08,male,3,1,2014.0,2014.0
107838,Mariana González Lago,Sociology,1.046114e+08,female,1,1,2015.0,2020.0
139415,Mariana Alejandra González,Sociology,1.226856e+08,female,0,1,2019.0,2019.0
139420,Mariana L. Gonzalez,Sociology,1.226856e+08,female,0,1,2012.0,2012.0
143860,Alejandra Mariana González,Sociology,1.536449e+08,female,0,1,2018.0,2018.0



— OpenAlex (DuckDB, ordered by citations):
OA query failed: Binder Error: Referenced column "last_known_institutions" not found in FROM clause!
Candidate bindings: "last_known_institution", "most_cited_work", "display_name_alternatives", "works_api_url", "works_count"

LINE 2: ...        SELECT id, display_name, works_count, cited_by_count, last_known_institutions
                                                                         ^
  (no matches)

— Pipeline (factuality_full.csv):


,model,language,role,task,location,field,subfield,author_status,field_status,seniority_status,location_status,affiliation_status,oa_id,oa_country_code,oa_last_institution
11884,gemini-2.5-flash,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Topology,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
13008,gemini-2.5-flash,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Number theory,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
14483,gemini-2.5-flash,english,Director/Recruiter,seeking potential hires,Ecuador,Sociology,Family,found,field_match,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
49682,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Physics,Education,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
49900,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Number theory,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
50252,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Physics,Condensed Matter,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
50417,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Biology,Neuroscience,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
50517,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Biology,Anatomy,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
51632,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Computer Science,Artificial Intelligence,found,field_mismatch,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet
52842,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Sociology,Criminology,found,field_match,seniority_unknown,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,DK,Rigshospitalet


In [6]:
print('Columnas SS:')
print(_df_ss.columns.tolist())
print()
print('Auto-detect:')
print(f'  _SS_NAME_COL     = {_SS_NAME_COL!r}')
print(f'  _SS_LASTNAME_COL = {_SS_LASTNAME_COL!r}')
print(f'  _SS_FULL_COL     = {_SS_FULL_COL!r}')
print()
print('Primeras 3 filas:')
_df_ss.head(3)
print(f'  _SS_FULL_COL     = {_SS_FULL_COL!r}')
print()
print('Primeras 3 filas:')
_df_ss.head(3)


Columnas SS:
['Researcher_id', 'First_year', 'Last_year', 'Ranking_position_citations', 'Citations', 'Productivity', 'Collaborations', 'Field', 'Gender', 'Name', 'Probability', 'Clean_standarized_name', 'Career_age', 'Number_of_genders', 'Gender_Namsor', 'Gender_probability_Namsor', 'Combined_gender']

Auto-detect:
  _SS_NAME_COL     = 'Name'
  _SS_LASTNAME_COL = None
  _SS_FULL_COL     = None

Primeras 3 filas:
  _SS_FULL_COL     = None

Primeras 3 filas:


,Researcher_id,First_year,Last_year,Ranking_position_citations,Citations,Productivity,Collaborations,Field,Gender,Name,Probability,Clean_standarized_name,Career_age,Number_of_genders,Gender_Namsor,Gender_probability_Namsor,Combined_gender
0,66098311.0,1964.0,2016.0,3,30575,84,33,Sociology,None,A. Giddens,NaN,None,55.0,0,None,0.0,None
1,97923787.0,1966.0,2019.0,17,13742,44,24,Sociology,male,Michel Foucault,0.74729,MICHEL,53.0,1,None,0.0,male
2,2906196.0,1972.0,2010.0,23,11975,40,14,Sociology,male,Michel Callon,0.74729,MICHEL,47.0,1,None,0.0,male


## Manual review step

Open `manual_validation_5.csv` in Excel/LibreOffice and fill in the empty columns for each persona row:

- `found_in_ss_manual`: `yes` / `no` / empty (unknown). Did you find this exact researcher on Semantic Scholar?
- `found_in_oa_manual`: same, for OpenAlex.
- `affiliation_correct_manual`: `yes` / `no` / empty. Does at least one institution in `current_affiliations` match a real affiliation of this researcher?
- `field_correct_manual`: `yes` / `no` / empty. Does the researcher actually work in the requested `field`?
- `notes_manual`: free text — record anything notable (e.g. "same name but different person", "affiliation outdated").

When done, re-run the cell below to compute concordance.

In [ ]:
# 5. Concordance after manual filling. Re-run after editing the CSV.
df_filled = pd.read_csv(OUT_CSV, dtype=str).fillna('')

def truthy(v):
    return str(v).strip().lower() in ('yes','y','true','1','t')

def normalize_manual_pair(ss, oa):
    if not ss and not oa:
        return None  # unknown
    return truthy(ss) or truthy(oa)

df_filled['manual_found_any'] = df_filled.apply(
    lambda r: normalize_manual_pair(r['found_in_ss_manual'], r['found_in_oa_manual']), axis=1)
df_filled['auto_found']       = df_filled['author_status_auto'].apply(
    lambda s: s != 'hallucinated' if s else None)

n_labeled    = df_filled['manual_found_any'].notna().sum()
n_compared   = df_filled.apply(lambda r: r['manual_found_any'] is not None and r['auto_found'] is not None, axis=1).sum()
if n_compared > 0:
    n_agree  = df_filled.apply(lambda r: r['manual_found_any'] is not None and r['auto_found'] is not None and r['manual_found_any'] == r['auto_found'], axis=1).sum()
    print(f'AUTHOR FOUND concordance: {n_agree}/{n_compared} = {100*n_agree/n_compared:.1f}%')
else:
    print('No manual labels filled yet — fill the CSV then re-run.')

# Per-request breakdown
if n_compared > 0:
    by_req = df_filled.groupby('request_id').apply(
        lambda g: pd.Series({
            'n_personas':     len(g),
            'labeled':        g['manual_found_any'].notna().sum(),
            'agree':          ((g['manual_found_any'] == g['auto_found']) & g['manual_found_any'].notna() & g['auto_found'].notna()).sum(),
        })
    )
    print()
    print(by_req)

# Affiliation concordance (when both sides present)
if 'affiliation_status_auto' in df_filled.columns and df_filled['affiliation_correct_manual'].str.strip().ne('').any():
    df_filled['auto_aff_correct']   = df_filled['affiliation_status_auto'].apply(
        lambda s: s == 'affiliation_match' if s else None)
    df_filled['manual_aff_correct'] = df_filled['affiliation_correct_manual'].apply(
        lambda s: truthy(s) if str(s).strip() else None)
    n_compared_aff = df_filled.apply(
        lambda r: r['auto_aff_correct'] is not None and r['manual_aff_correct'] is not None, axis=1).sum()
    if n_compared_aff > 0:
        n_agree_aff = df_filled.apply(
            lambda r: r['auto_aff_correct'] is not None and r['manual_aff_correct'] is not None and r['auto_aff_correct'] == r['manual_aff_correct'], axis=1).sum()
        print(f'\nAFFILIATION concordance: {n_agree_aff}/{n_compared_aff} = {100*n_agree_aff/n_compared_aff:.1f}%')